# softmax的简单实现

## 初始化模型参数

pytorch不会隐式地调整输入的形状。在线性层之前，我们需要定义展平层，调整网络的输入形状

In [2]:
import torch
from torch import nn
from d2l import torch as d2l
batch_size = 256
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)

net = nn.Sequential(nn.Flatten(), nn.Linear(784, 10))  # 输入是28*28的图片，flatten层把图片展平为784维的一维向量
# nn.Linear()：全连接层，执行矩阵的乘法计算

def init_weights(m): # m参数代表模型中的一个层
    if type(m) == nn.Linear:  # 如果是全连接层
        nn.init.normal_(m.weight, std=0.01)  # 结尾的_表示在原地修改参数normal_(tensor(参数)， mean, std)

net.apply(init_weights)  # 把init_weights函数应用到模型中的每个层


Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=10, bias=True)
)

## softmax的实现

在计算softmax的时候，先从所有的输出中减去最大的输出，再计算指数函数，最后除以指数函数的和。

$$
\hat{y_j} = \frac{exp{(o_j)}} {\sum_{k}exp{(o_k)}}
$$

设置$ c = \max(o_j) $,则原式子可以化为：
$$\frac{\exp(o_j)}{\sum \exp(o_k)} = \frac{\exp(o_j - c) \cdot \exp(c)}{\sum (\exp(o_k - c) \cdot \exp(c))} = \frac{\exp(o_j - c)}{\sum \exp(o_k - c)}$$

主要是防止指数的溢出问题

In [3]:
loss = nn.CrossEntropyLoss(reduction='none')

reduction是损失函数中控制损失聚合方式的参数：\
none: 不聚和，保留每个样本的损失值，返回形状为(batch_size,) \
mean: 返回样本损失的平均值 \
sum：返回所有样本损失的总和